# Exercise 4: State Management & Preferences

## Learning Objectives

In this exercise, you will:
- Understand state prefixes (`user:`, `temp:`, `app:`) for different persistence scopes
- Implement preference persistence across conversation turns
- Use state injection in agent instructions
- Build a budget-aware travel assistant that remembers user preferences

**Estimated time:** 15-20 minutes

## What is State Management?

**You've already used sessions!** In Exercises 1-3, you used `InMemorySessionService` to manage conversations.

Now we'll learn about **state prefixes** - the magic that controls how long data persists.

---

### State Prefixes: The Key Insight

| Prefix | Scope | Persists | Use Case |
|--------|-------|----------|----------|
| No prefix | Session | Current conversation only | Search results, current step |
| `user:` | User | Across all user sessions | Budget, travel style, dietary restrictions |
| `temp:` | Invocation | Current tool call only | API responses, intermediate calculations |
| `app:` | Application | Shared across all users | Feature flags, app version |

**The key insight:** The prefix controls persistence, not the SessionService!

```python
# Session-scoped (lost when new session starts)
context.state["last_search"] = "Tokyo flights"

# User-scoped (persists across sessions!)
context.state["user:budget"] = 1000

# Temp-scoped (only for this tool call)
context.state["temp:api_response"] = {...}
```

---

### Why This Matters for Travel Agents

**Without state prefixes:**
```
Session 1: "My budget is $1000" -> Agent remembers
Session 2: "Find flights to Tokyo" -> Agent: "What's your budget?" (forgot!)
```

**With `user:` prefix:**
```
Session 1: "My budget is $1000" -> Saved as user:budget = 1000
Session 2: "Find flights to Tokyo" -> Agent: "Here are flights under $1000!" (remembered!)
```

> Remember: You're building context that flows through conversations - that's context engineering!

## Setup Instructions

**Estimated time:** ~2 minutes

This exercise assumes you have completed Exercises 1-3.

**Authentication:** This exercise uses the simple API key approach (like Exercises 1-2), not Vertex AI.

In [ ]:
# @title Quick Setup (Run this cell first)
# This cell sets up your environment - you can collapse it after running

import os
import google.generativeai as genai
from getpass import getpass

# Step 1: Install Google ADK
!pip install -q google-adk==1.23.0
print("Google ADK 1.23.0 installed")

# Step 2: Configure API Key
api_key = getpass('Enter your Google AI API Key: ')
genai.configure(api_key=api_key)
os.environ['GOOGLE_API_KEY'] = api_key
print("API Key configured")

print("\nReady to learn state management!")

In [ ]:
# Import ADK components for running agents
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.context import ToolContext
from google.genai.types import Content, Part
from datetime import datetime
from typing import Optional

# Create session service (you've used this before!)
session_service = InMemorySessionService()
user_id = "workshop_participant"

print("ADK components imported")
print("Session service created")
print("ToolContext imported (for state management in tools)")

---

## Exercise 4A: Basic State Usage

**Estimated time:** ~5 minutes

<!-- INSTRUCTOR: Start simple - show state without prefixes first -->

First, let's see how basic session state works (without prefixes).

### Your Tasks:

1. Create a tool that stores information in session state
2. Create another tool that reads from session state
3. Observe what happens when you create a new session

In [ ]:
# Exercise 4A: Basic state tool (no prefix = session-scoped)

def save_note(context: ToolContext, note: str) -> str:
    """
    Save a note to session state.
    
    Args:
        context: Tool context with access to session state
        note: The note to save
    
    Returns:
        Confirmation message
    """
    # TODO: Save the note to session state
    # Hint: context.state["key"] = value
    # Use key "last_note" (no prefix = session-scoped)
    print(f"  [DEBUG] Saving note: {note}")
    
    # YOUR CODE HERE:
    # context.state["last_note"] = note
    
    return f"Note saved: '{note}'"


def get_note(context: ToolContext) -> str:
    """
    Retrieve the saved note from session state.
    
    Args:
        context: Tool context with access to session state
    
    Returns:
        The saved note or a message if none exists
    """
    # TODO: Read the note from session state
    # Hint: context.state.get("key", default_value)
    print("  [DEBUG] Retrieving note...")
    
    # YOUR CODE HERE:
    # note = context.state.get("last_note")
    note = None  # Replace with state read
    
    if note:
        return f"Your note: '{note}'"
    return "No note saved yet."

print("Note tools defined")

In [ ]:
# Create simple note agent
note_agent = Agent(
    model='gemini-2.5-flash',
    name='note_agent',
    description='Simple agent that saves and retrieves notes',
    instruction='''You are a helpful assistant that can save and retrieve notes.

When user says "remember" or "save", use save_note tool.
When user asks "what did I say" or "my note", use get_note tool.

Be brief and helpful.''',
    tools=[save_note, get_note],
)

print("Note agent created")

In [ ]:
# Helper function for testing
async def run_query(agent, session, query):
    """Run a single query and return the response."""
    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )
    
    print(f"\nYou: {query}")
    
    final_response = ""
    async for event in runner.run_async(
        user_id=user_id,
        session_id=session.id,
        new_message=Content(parts=[Part(text=query)], role="user")
    ):
        if event.is_final_response():
            if event.content and hasattr(event.content, 'parts') and event.content.parts:
                final_response = event.content.parts[0].text
            break
    
    print(f"Agent: {final_response}")
    return final_response

In [ ]:
# Test 1: Save and retrieve in SAME session
print("="*60)
print("Test 1: Same Session")
print("="*60)

# Create session 1
session1 = await session_service.create_session(
    app_name=note_agent.name,
    user_id=user_id
)
print(f"Session 1 created: {session1.id[:8]}...")

# Save a note
await run_query(note_agent, session1, "Remember this: my favorite color is blue")

# Retrieve in same session
await run_query(note_agent, session1, "What did I tell you to remember?")

# Expected: Note should be retrieved successfully

In [ ]:
# Test 2: Try to retrieve in NEW session
print("\n" + "="*60)
print("Test 2: New Session (state should be LOST!)")
print("="*60)

# Create session 2 (NEW session)
session2 = await session_service.create_session(
    app_name=note_agent.name,
    user_id=user_id  # Same user, different session
)
print(f"Session 2 created: {session2.id[:8]}...")

# Try to retrieve
await run_query(note_agent, session2, "What did I tell you to remember?")

# Expected: "No note saved yet" - state was lost when session changed!

### Checkpoint: Session State Lost

You should see:
- **Test 1**: Note saved and retrieved successfully (same session)
- **Test 2**: "No note saved" - state was lost!

**What happened?**

Without a prefix, state is **session-scoped**. When you created a new session, the state was gone!

This is a problem for user preferences. Users don't want to re-enter their budget every conversation.

**Solution:** Use the `user:` prefix for cross-session persistence.

---

## Exercise 4B: User Preferences with `user:` Prefix

**Estimated time:** ~7 minutes

<!-- INSTRUCTOR: This is the key insight - user: prefix for cross-session -->

Now let's use the `user:` prefix to persist preferences across sessions.

### Your Tasks:

1. Create `remember_preference` tool using `user:` prefix
2. Create agent with state injection in instruction
3. Test that preferences persist across sessions

In [ ]:
# Exercise 4B: Preference tools with user: prefix

def remember_preference(context: ToolContext, preference_type: str, value: str) -> str:
    """
    Store user preference using user: prefix for cross-session persistence.
    
    Args:
        context: Tool context with access to session state
        preference_type: Type of preference (budget, travel_style, dietary_restrictions)
        value: The preference value to store
    
    Returns:
        Confirmation message
    """
    # TODO: Store with user: prefix so it persists across sessions
    # Hint: context.state[f"user:{preference_type}"] = value
    print(f"  [DEBUG] Saving preference: user:{preference_type} = {value}")
    
    # YOUR CODE HERE:
    # context.state[f"user:{preference_type}"] = value
    
    return f"I'll remember that your {preference_type} is {value}"


def get_preference(context: ToolContext, preference_type: str) -> str:
    """
    Retrieve stored preference.
    
    Args:
        context: Tool context with access to session state
        preference_type: Type of preference to retrieve
    
    Returns:
        The preference value or message if not set
    """
    # TODO: Read from user:-prefixed key
    print(f"  [DEBUG] Reading preference: user:{preference_type}")
    
    # YOUR CODE HERE:
    # value = context.state.get(f"user:{preference_type}")
    value = None  # Replace with state read
    
    if value:
        return f"Your {preference_type} is {value}"
    return f"No {preference_type} preference set yet."

print("Preference tools defined (with user: prefix)")

In [ ]:
# Create preference agent with state injection in instruction

preference_agent = Agent(
    model='gemini-2.5-flash',
    name='preference_agent',
    description='Travel assistant that remembers user preferences',
    
    # TODO: Notice the state injection syntax: {user:key?}
    # The ? makes it optional (no error if key doesn't exist)
    instruction='''You are a travel assistant that remembers preferences.

Current user preferences (from previous conversations):
- Budget: {user:budget?}
- Travel style: {user:travel_style?}
- Dietary restrictions: {user:dietary_restrictions?}

When user mentions a preference, use remember_preference to save it.
When user asks about their preferences, use get_preference to retrieve them.

If a preference is not set (empty above), mention that you don't have it yet.

Be friendly and helpful!''',
    
    tools=[remember_preference, get_preference],
)

print("Preference agent created")
print("State injection syntax: {user:budget?} injects state into instruction")

In [ ]:
# Test: Set preference in Session 1
print("="*60)
print("Test: Session 1 - Set Preferences")
print("="*60)

session_a = await session_service.create_session(
    app_name=preference_agent.name,
    user_id=user_id
)
print(f"Session A created: {session_a.id[:8]}...")

# Set budget preference
await run_query(preference_agent, session_a, "My budget for travel is $1500")

# Set travel style
await run_query(preference_agent, session_a, "I prefer luxury travel")

# Verify in same session
await run_query(preference_agent, session_a, "What preferences do you know about me?")

In [ ]:
# Test: Retrieve preferences in NEW Session 2
print("\n" + "="*60)
print("Test: Session 2 - Preferences Should Persist!")
print("="*60)

session_b = await session_service.create_session(
    app_name=preference_agent.name,
    user_id=user_id  # Same user, NEW session
)
print(f"Session B created: {session_b.id[:8]}...")

# Ask about preferences (should still know them!)
await run_query(preference_agent, session_b, "What's my budget?")
await run_query(preference_agent, session_b, "What's my travel style?")

### Checkpoint: Preferences Persist!

You should see:
- **Session A**: Preferences saved (budget = $1500, style = luxury)
- **Session B**: Preferences still remembered in a NEW session!

**What made the difference?**

Using `user:` prefix: `context.state["user:budget"]` instead of `context.state["budget"]`

The `user:` prefix tells ADK to persist this data across all sessions for this user.

**State injection:**
- `{user:budget?}` in the instruction automatically injects the saved budget
- The `?` makes it optional (no error if not set yet)
- The agent sees this value in its instruction every turn

### Solution: Complete Preference Tools

In [ ]:
# @title Solution: Exercise 4B (Expand to see)

def remember_preference_solution(context: ToolContext, preference_type: str, value: str) -> str:
    """
    Store user preference using user: prefix for cross-session persistence.
    
    Args:
        context: Tool context with access to session state
        preference_type: Type of preference (budget, travel_style, dietary_restrictions)
        value: The preference value to store
    
    Returns:
        Confirmation message
    """
    # Store with user: prefix so it persists across sessions
    context.state[f"user:{preference_type}"] = value
    print(f"  [SOLUTION] Saved user:{preference_type} = {value}")
    return f"I'll remember that your {preference_type} is {value}"


def get_preference_solution(context: ToolContext, preference_type: str) -> str:
    """
    Retrieve stored preference.
    
    Args:
        context: Tool context with access to session state
        preference_type: Type of preference to retrieve
    
    Returns:
        The preference value or message if not set
    """
    # Read from user:-prefixed key
    value = context.state.get(f"user:{preference_type}")
    print(f"  [SOLUTION] Read user:{preference_type} = {value}")
    
    if value:
        return f"Your {preference_type} is {value}"
    return f"No {preference_type} preference set yet."

print("Solution tools defined")

---

## Exercise 4C: Multi-Turn Conversation with Preferences

**Estimated time:** ~5 minutes

<!-- INSTRUCTOR: This demonstrates the full value - preferences applied automatically -->

Now let's build a complete travel assistant that:
1. Remembers user preferences (budget, style)
2. Auto-applies preferences to searches
3. Updates preferences when user changes their mind

### Your Task:

Follow the multi-turn conversation and observe how preferences flow through.

In [ ]:
# Complete travel agent with preferences and booking

# Booking tools (from Exercise 2)
def search_flights(
    origin: str,
    destination: str,
    departure_date: str,
    passengers: int = 1,
    max_price: Optional[int] = None,
    context: ToolContext = None
) -> dict:
    """
    Search for available flights. Auto-applies saved budget if max_price not specified.
    
    Args:
        origin: Departure airport code (e.g., 'SFO')
        destination: Arrival airport code (e.g., 'NRT')
        departure_date: Date in YYYY-MM-DD format
        passengers: Number of passengers (default 1)
        max_price: Maximum price per person (optional - uses saved budget if not set)
        context: Tool context for state access
    
    Returns:
        Dictionary with flight data
    """
    print(f"  [TOOL] search_flights: {origin} -> {destination}")
    
    # TODO: Auto-apply saved budget if max_price not specified
    # Hint: Check context.state.get("user:budget") if context exists
    if max_price is None and context:
        saved_budget = context.state.get("user:budget")
        if saved_budget:
            try:
                # Remove $ and convert to int
                max_price = int(str(saved_budget).replace('$', '').replace(',', ''))
                print(f"  [TOOL] Using saved budget: ${max_price}")
            except:
                pass
    
    # Mock flight data
    all_flights = [
        {"airline": "United", "price": 850, "departure": "08:30", "duration": "11h"},
        {"airline": "ANA", "price": 1200, "departure": "11:00", "duration": "11h"},
        {"airline": "JAL", "price": 1100, "departure": "13:45", "duration": "11h"},
        {"airline": "Delta", "price": 780, "departure": "15:30", "duration": "12h"},
    ]
    
    # Filter by budget if specified
    if max_price:
        flights = [f for f in all_flights if f["price"] <= max_price]
        print(f"  [TOOL] Filtered to {len(flights)} flights under ${max_price}")
    else:
        flights = all_flights
    
    return {
        "status": "success",
        "flights": flights,
        "route": f"{origin} -> {destination}",
        "budget_applied": max_price
    }


def search_hotels(
    location: str,
    check_in: str,
    check_out: str,
    guests: int = 1,
    max_price_per_night: Optional[int] = None,
    context: ToolContext = None
) -> dict:
    """
    Search for available hotels. Auto-applies saved travel style preference.
    
    Args:
        location: City name (e.g., 'Tokyo')
        check_in: Check-in date in YYYY-MM-DD format
        check_out: Check-out date in YYYY-MM-DD format
        guests: Number of guests (default 1)
        max_price_per_night: Maximum price per night (optional)
        context: Tool context for state access
    
    Returns:
        Dictionary with hotel data
    """
    print(f"  [TOOL] search_hotels: {location}")
    
    # Check saved travel style
    travel_style = None
    if context:
        travel_style = context.state.get("user:travel_style")
        if travel_style:
            print(f"  [TOOL] Using saved travel style: {travel_style}")
    
    # Mock hotel data
    all_hotels = [
        {"name": "Park Hyatt Tokyo", "stars": 5, "price_per_night": 450, "style": "luxury"},
        {"name": "Shinjuku Granbell", "stars": 4, "price_per_night": 180, "style": "moderate"},
        {"name": "Hostel Chapter Two", "stars": 2, "price_per_night": 45, "style": "budget"},
        {"name": "MUJI Hotel Ginza", "stars": 4, "price_per_night": 220, "style": "moderate"},
        {"name": "Aman Tokyo", "stars": 5, "price_per_night": 800, "style": "luxury"},
    ]
    
    # Filter by travel style if saved
    hotels = all_hotels
    if travel_style:
        style_lower = travel_style.lower()
        if 'luxury' in style_lower:
            hotels = [h for h in all_hotels if h["style"] == "luxury"]
        elif 'budget' in style_lower:
            hotels = [h for h in all_hotels if h["style"] == "budget"]
    
    # Filter by price if specified
    if max_price_per_night:
        hotels = [h for h in hotels if h["price_per_night"] <= max_price_per_night]
    
    return {
        "status": "success",
        "hotels": hotels,
        "location": location,
        "travel_style_applied": travel_style
    }

print("Booking tools defined (with preference auto-apply)")

In [ ]:
# Create complete travel assistant

travel_assistant = Agent(
    model='gemini-2.5-flash',
    name='travel_assistant',
    description='Travel assistant with preference memory and booking tools',
    
    instruction='''You are a travel booking assistant that remembers user preferences.

SAVED USER PREFERENCES:
- Budget: {user:budget?}
- Travel style: {user:travel_style?}
- Dietary restrictions: {user:dietary_restrictions?}

YOUR TOOLS:
- remember_preference: Save user preferences for future conversations
- get_preference: Retrieve saved preferences
- search_flights: Find flights (auto-applies saved budget)
- search_hotels: Find hotels (auto-applies saved travel style)

INSTRUCTIONS:
1. When user mentions a preference (budget, style, dietary), save it immediately
2. When searching, apply saved preferences automatically
3. If preferences are empty, ask user about them
4. Show only options that match their preferences

Be helpful, remember their preferences, and make travel planning easy!''',
    
    tools=[remember_preference, get_preference, search_flights, search_hotels],
)

print("Travel assistant created with preference memory")

In [ ]:
# Multi-turn conversation test
print("="*60)
print("Multi-Turn Conversation: Preferences Flow Through")
print("="*60)

# Create session
conversation_session = await session_service.create_session(
    app_name=travel_assistant.name,
    user_id=user_id
)

# Turn 1: Set budget
print("\n--- Turn 1: Set Budget ---")
await run_query(travel_assistant, conversation_session, 
    "My budget is $1000 for this trip")

In [ ]:
# Turn 2: Search flights (should auto-apply budget)
print("\n--- Turn 2: Search Flights (budget should auto-apply) ---")
await run_query(travel_assistant, conversation_session,
    "Find me flights from San Francisco to Tokyo on March 15, 2026")

# Expected: Only flights under $1000 shown

In [ ]:
# Turn 3: Set travel style
print("\n--- Turn 3: Set Travel Style ---")
await run_query(travel_assistant, conversation_session,
    "I prefer luxury travel")

In [ ]:
# Turn 4: Search hotels (should auto-apply travel style)
print("\n--- Turn 4: Search Hotels (style should auto-apply) ---")
await run_query(travel_assistant, conversation_session,
    "Find me hotels in Tokyo for March 15-20")

# Expected: Only luxury hotels shown

In [ ]:
# Turn 5: Update preference
print("\n--- Turn 5: Update Budget ---")
await run_query(travel_assistant, conversation_session,
    "Actually, I can spend up to $1500")

In [ ]:
# Turn 6: New search with updated budget
print("\n--- Turn 6: Search Again (new budget should apply) ---")
await run_query(travel_assistant, conversation_session,
    "Show me flights again")

# Expected: More flight options (up to $1500)

### Checkpoint: Multi-Turn Context Working

You should see:
- **Turn 1**: Agent saves budget preference ($1000)
- **Turn 2**: Flight search auto-applies $1000 budget filter
- **Turn 3**: Agent saves travel style (luxury)
- **Turn 4**: Hotel search auto-applies luxury filter
- **Turn 5**: Agent updates budget to $1500
- **Turn 6**: New budget reflected in flight results

**What just happened?**
1. Budget stored with `user:budget` prefix
2. Tool functions read from `context.state.get("user:budget")`
3. Budget applied even without user re-stating it
4. Preference updates immediately take effect

**This is context engineering:** relevant state flows through the conversation automatically!

In [ ]:
# Bonus: Verify preferences persist to NEW session
print("\n" + "="*60)
print("Bonus: New Session - Preferences Should Still Work!")
print("="*60)

new_session = await session_service.create_session(
    app_name=travel_assistant.name,
    user_id=user_id
)
print(f"New session created: {new_session.id[:8]}...")

await run_query(travel_assistant, new_session,
    "What are my preferences?")

# Expected: Budget $1500, travel style luxury still known!

---

## Challenge (Optional)

Ready for more? Try these challenges:

### Challenge 1: Add More Preference Types

Add support for:
- `user:dietary_restrictions` (vegetarian, halal, etc.)
- `user:preferred_airlines` (United, Delta, etc.)
- `user:seat_preference` (window, aisle)

### Challenge 2: Clear Preferences

Add a `clear_preference` tool that removes a saved preference:
```python
def clear_preference(context: ToolContext, preference_type: str) -> str:
    del context.state[f"user:{preference_type}"]
    return f"Cleared your {preference_type} preference"
```

### Challenge 3: Preference Conflict Detection

What if budget is $100 but travel style is luxury?
- Detect the conflict
- Alert the user
- Suggest adjustments

### Challenge 4: App-Wide Settings

Try using `app:` prefix for settings shared across all users:
```python
context.state["app:default_currency"] = "USD"
context.state["app:max_search_results"] = 5
```

In [ ]:
# Your challenge code here
# TODO: Implement additional preference types
# TODO: Add clear_preference tool
# TODO: Add conflict detection

---

## What You Learned

**Congratulations!** You've mastered state management in ADK!

### Key Concepts

**State Prefixes:**
- No prefix = session-scoped (current conversation only)
- `user:` = user-scoped (persists across sessions)
- `temp:` = invocation-scoped (current tool call only)
- `app:` = app-scoped (shared across all users)

**State Injection:**
- Use `{user:budget?}` in instruction to inject state
- The `?` makes the key optional (no error if missing)
- Agent sees current state values every turn

**Context Engineering:**
- Store preferences explicitly with `remember_preference`
- Inject state into instructions for agent awareness
- Auto-apply preferences in tool functions
- Build conversations where context flows naturally

### Patterns You Built

1. **Basic state**: Read/write session state (Exercise 4A)
2. **User preferences**: Cross-session persistence with `user:` prefix (Exercise 4B)
3. **Multi-turn context**: Preferences auto-applied across conversation (Exercise 4C)

### Production Considerations

**Workshop used:**
- `InMemorySessionService` (state lost on restart)

**Production would use:**
- `DatabaseSessionService` (PostgreSQL, MySQL, SQLite)
- `VertexAiSessionService` (Vertex AI Agent Engine)

The state prefix patterns (`user:`, `temp:`, `app:`) work identically regardless of SessionService!

---

### What's Next?

You now have a complete ADK travel agent with:
- Hello World agent (Exercise 1)
- Function calling tools for bookings (Exercise 2)
- RAG knowledge retrieval for destinations (Exercise 3)
- State management for user preferences (Exercise 4)

**Resources:**
- [ADK State Management Docs](https://google.github.io/adk-docs/sessions/state/)
- [Session Services Reference](https://google.github.io/adk-docs/sessions/)
- [DatabaseSessionService Guide](https://google.github.io/adk-docs/sessions/database-session-service/)